# Comparing two unpaired means 
## Introduction


## Preparing data for hypothesis testing
### Descriptive statistics and visualization


In [ ]:
import numpy as np

# We use the data from Table 30.1 of the book Intuitive Biostatistics
old = np.array([20.8, 2.8, 50, 33.3, 29.4, 38.9, 29.4, 52.6, 14.3])  # old
yng = np.array([45.5, 55, 60.7, 61.5, 61.1, 65.5, 42.9, 37.5])       # young

In [ ]:
import pandas as pd
import scipy.stats as stats

# Descriptive statistics, presentation enhanced via Series
old_stats = stats.describe(old)
old_stats_series = pd.Series(old_stats._asdict())

yng_stats = stats.describe(yng)
yng_stats_series = pd.Series(yng_stats._asdict())

print("Descriptive statistics for old rats:")
print(old_stats_series)
print()  # print blank separation line
print("Descriptive statistics for young rats:")
print(yng_stats_series)

In [ ]:

import matplotlib.pyplot as plt
import seaborn as sns

# Box and swarm plots
plt.figure(figsize=(3.5, 3))
sns.boxplot(data=[old, yng])
sns.swarmplot(data=[old, yng], color='black')
plt.ylabel(r"%Emax")
plt.xticks([0, 1], ['old', 'young'])
plt.title("Box and swarm plots");

### Assessing assumptions
#### Normality testing


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pingouin as pg

# Function to perform tests and print results
def normality_tests(data, sample_type):
    s = f"{'Test':<20} {'W':<6} {'P value':<8} {'normal'}"
    print(f"Normality tests for {sample_type}:")
    print(s)

    # D'Agostino-Pearson Test
    dagostino_results = pg.normality(data, method='normaltest')
    print(f"{'D\'Agostino-Pearson':<20} {dagostino_results['W'][0]:<6.2f} \
{dagostino_results['pval'][0]:<8.3f} {dagostino_results['normal'][0]}")

    # Shapiro-Wilk Test
    shapiro_results = pg.normality(
        data,
        method='shapiro')
    print(f"{'Shapiro-Wilk':<20} {shapiro_results['W'][0]:<6.2f} \
{shapiro_results['pval'][0]:<8.3f} {shapiro_results['normal'][0]}")

In [ ]:
# Perform tests and print results
normality_tests(old, "old rats")
print("")
normality_tests(yng, "young rats")

In [ ]:

# Plotting Q-Q plots
_, axes = plt.subplots(nrows=1, ncols=2, figsize=(6, 3))

for i, (data, title) in enumerate(
    zip([old, yng], ["Old rats", "Young rats"])):
    pg.qqplot(
        data,
        dist='norm',
        ax=axes[i])
    axes[i].set_title(title)

plt.tight_layout();

#### Homoscedasticity testing
##### F-test of equality of variances


In [ ]:
# Calculate sample variances s (unbiased estimator)
var_old = np.var(old, ddof=1)
var_yng = np.var(yng, ddof=1)

# Calculate the ratio of variances
var_ratio = var_old / var_yng

# Sample sizes
n_old = len(old)
n_yng = len(yng)

# Degrees of freedom
df_old = n_old - 1
df_yng = n_yng - 1

# Calculate the F-statistic
f_statistic = max(var_ratio, 1/var_ratio)
# Ensures that the F-statistic >= 1

# Calculate the P value using the F-distribution (two-sided test)
p_value_fstat = 2 * (1 - stats.f.cdf(f_statistic, df_old, df_yng))

# Print the results
print(f"Variance ratio (old:young) = {var_ratio:.3f}")
print(f"F-statistic (ratio >= 1) = {f_statistic:.3f}")
print(f"P value for the F-test = {p_value_fstat:.4f}")

In [ ]:

# Significance level (alpha)
α = 0.05

# Calculate critical value
f_crit = stats.f.ppf(1 - α, df_old, df_yng)

# Generate x values for plotting
x = np.linspace(0, 10, 100)
hx = stats.f.pdf(x, df_old, df_yng)

# Create the plot
plt.figure(figsize=(3.5, 3))
plt.plot(x, hx, lw=2, color="black")

# Plot the critical value
plt.axvline(
    x=f_crit,
    color='orangered', linestyle='--',
    label=f'F* ({f_crit:.3f})')

# Shade the probability alpha
plt.fill_between(
    x[x >= f_crit], hx[x >= f_crit],
    linestyle="-",  linewidth=2, color='tomato',
    label=f'α ({α})')

# Plot the observed F-statistic
plt.axvline(
    x=f_statistic,
    color='limegreen', linestyle='-.',
    label=f'F ({f_statistic:.3f})')

# Shade the P value area
plt.fill_between(
    x[x >= f_statistic], hx[x >= f_statistic],
    color='greenyellow',
    label=f'P ({p_value_fstat:.3f})')

# Add labels and title
plt.xlabel('F')
plt.ylabel('Density')
plt.margins(x=0, y=0)
plt.yticks([])
plt.title(f"F-distribution (DFn={df_old}, DFd={df_yng})")
plt.legend();

##### Levene and Bartlett tests


In [ ]:
# Using SciPy
# Levene's test for equal variances
levene_statistic, levene_p_value = stats.levene(old, yng)

# Bartlett's test for equal variances
bartlett_statistic, bartlett_p_value = stats.bartlett(old, yng)

# Print the results
print("Homoscedasticity tests (SciPy):")
print(f"  Levene's test statistic = {levene_statistic:.3f}, \
P value = {levene_p_value:.4f}")
print(f"  Bartlett's test statistic = {bartlett_statistic:.3f}, \
P value = {bartlett_p_value:.4f}")

In [ ]:
# Using Pingouin
# Levene's test for equal variances
levene_results = pg.homoscedasticity([old, yng], method='levene')

# Bartlett's test for equal variances
bartlett_results = pg.homoscedasticity([old, yng], method='bartlett')

# Print the results
print("Homoscedasticity tests (Pingouin):")
print(levene_results)
print("")
print(bartlett_results)

## The t-test


In [ ]:
mean_diff = np.abs(np.mean(yng) - np.mean(old))
print(
    f"Absolute mean difference between old and young rats = {mean_diff:.3f}"
)

### The t-ratio
### Welch's t-test
#### Manual calculation of the Welch t-ratio


In [ ]:
# Calculate the standard error for unequal variances (Welch's t-test)
# The variances were already calculated for the F-test
se_unequal = np.sqrt(var_old / n_old + var_yng / n_yng)

# Calculate Welch's t-statistic
t_statistic_welch = mean_diff / se_unequal

# Calculate the degrees of freedom using the Welch-Satterthwaite equation
"""
We haven't mentioned it yet, but parentheses are a useful notation in Python.
When opened, they allow us to wrap calculation elements, making them easier 
to enter and view. We will use them frequently during long calculations 
and with DataFrames.
"""
df_welch = (
    se_unequal**4 /
    (   # se^2 = s/n --> se^4 = (s/n)^2
        (var_old / n_old)**2 / (n_old - 1)
        +
        (var_yng / n_yng)**2 / (n_yng - 1)
    ))

# Print the results
print(f"Welch's t-statistic = {t_statistic_welch:.2f} with {df_welch:.3f} \
degrees of freedom")
print("(Welch-Satterthwaite approximation)")

#### Welch's P value


In [ ]:
# Calculate the P value using the t-distribution (two-sided test)
p_value_welch = 2 * (1 - stats.t.cdf(abs(t_statistic_welch), df_welch))

# Print the results
print(f"P value for Welch's t-test (two-sided) = {p_value_welch:.5f}")

#### Visualizing Welch's values


In [ ]:

# Calculate critical t-values (two-tailed test)
t_crit_lower_welch = stats.t.ppf(α/2, df_welch)
t_crit_upper_welch = stats.t.ppf(1 - α/2, df_welch)

# Generate x values for plotting
x = np.linspace(-5, 5, 1000)
hx = stats.t.pdf(x, df_welch)

# Create the plot
plt.figure(figsize=(3.5, 3))
plt.plot(x, hx, lw=2, color="black")

# Plot the critical t-values
plt.axvline(
    x=t_crit_upper_welch,
    color='orangered', linestyle='--',
    label=f't* ({t_crit_upper_welch:.3f})')
plt.axvline(
    x=t_crit_lower_welch,
    color='orangered', linestyle='--',)

# Shade the rejection regions (alpha)
plt.fill_between(
    x[x <= t_crit_lower_welch],
    hx[x <= t_crit_lower_welch],
    linestyle="-", linewidth=2, color='tomato',
    label=f'α ({α})')
plt.fill_between(
    x[x >= t_crit_upper_welch],
    hx[x >= t_crit_upper_welch],
    linestyle="-", linewidth=2,
    color='tomato')

# Plot the observed t-statistic
plt.axvline(
    x=t_statistic_welch,
    color='limegreen', linestyle='-.',
    label=f't ({t_statistic_welch:.3f})')

# Shade the P value areas (two-tailed)
plt.fill_between(
    x[x <= -abs(t_statistic_welch)],
    hx[x <= -abs(t_statistic_welch)],
    color='greenyellow',
    label=fr'P ({p_value_welch:.3f})',)
plt.fill_between(
    x[x >= abs(t_statistic_welch)],
    hx[x >= abs(t_statistic_welch)],
    color='greenyellow')

# Add labels and title
plt.xlabel('t')
plt.ylabel('Density')
plt.margins(x=0, y=0)
plt.yticks([])
plt.title(f"t-distribution (DF={df_welch:.2f})")
plt.legend();

#### Welch's confidence interval


In [ ]:
# Calculate the confidence interval (e.g., 1-α = 95% confidence)
margin_of_error_welch = abs(
    stats.t.ppf((1 + (1 - α)) / 2, df_welch)) * se_unequal

ci_welch = np.array([
    mean_diff - margin_of_error_welch,
    mean_diff + margin_of_error_welch])

# Print the results
print(
    r"95% CI for the mean difference (Welch):",
    ci_welch.round(3))

#### Performing the Welch's t-test in Python


In [ ]:
# Welch's t-test using SciPy
t_statistic_scipy_welch, p_value_scipy_welch = stats.ttest_ind(
    yng, old, equal_var=False)

# Welch's t-test using statsmodels
import statsmodels.api as sm

t_statistic_sm_welch, p_value_sm_welch, df_sm_welch = sm.stats.ttest_ind(
    yng, old, usevar='unequal')

# Welch's t-test using Pingouin
ttest_results_pingouin_welch = pg.ttest(
    yng, old, correction=True)

# Print all the results
print("Welch's t-test results (unequal variances):")
print(f"t-statistic (SciPy) = {t_statistic_scipy_welch:.3f}, \
P value = {p_value_scipy_welch:.4f}")
print(f"t-statistic (statsmodels) = {t_statistic_sm_welch:.3f}, \
P value = {p_value_sm_welch:.4f}, DF = {df_sm_welch:.3f}")
print()
print("Welch's t-test summary table (Pingouin):")
print(ttest_results_pingouin_welch.round(3))

### Student t-test
#### Student's t-ratio formula


In [ ]:
# Degrees of freedom for equal variances t-test
df_student = n_old + n_yng - 2

# Calculate pooled variance using sample variances 
# determined in the previous section
pooled_var = (
    (n_old - 1) * var_old + (n_yng - 1) * var_yng) / df_student

# Calculate the pooled standard error
se_pooled = np.sqrt(pooled_var * (1/n_old + 1/n_yng))

# Calculate the Student's t-statistic
t_statistic_student = mean_diff / se_pooled

# Print the results
print(f"Student's t-statistic = {t_statistic_student:.4f} with \
{df_student} degrees of freedom")

#### Special case of equal variances and sample sizes


#### Student's P value


In [ ]:
# Calculate the P value using the t-distribution (two-sided test)
p_value_student = 2*(1 - stats.t.cdf(abs(t_statistic_student), df_student))
# Print the results
print(f"P value for the Student's t-test = {p_value_student:.5f}")

#### Visualizing Student's values


In [ ]:

# Calculate critical t-values (two-tailed test)
t_crit_lower_student = stats.t.ppf(α/2, df_student)
t_crit_upper_student = stats.t.ppf(1 - α/2, df_student)

# Generate x values for plotting
x = np.linspace(-5, 5, 1000)
hx = stats.t.pdf(x, df_student)

# Create the plot
plt.figure(figsize=(3.5, 3))
plt.plot(x, hx, lw=2, color="black")

# Plot the critical t-values
plt.axvline(
    x=t_crit_lower_student,
    color='orangered', linestyle='--',)

plt.axvline(
    x=t_crit_upper_student,
    color='orangered', linestyle='--',
    label=f't* ({t_crit_upper_student:.3f})')

# Shade the rejection regions (alpha)
plt.fill_between(
    x[x <= t_crit_lower_student],
    hx[x <= t_crit_lower_student],
    linestyle="-", linewidth=2, color='tomato',
    label=f'α ({α})')
plt.fill_between(
    x[x >= t_crit_upper_student],
    hx[x >= t_crit_upper_student],
    linestyle="-", linewidth=2,
    color='tomato')

# Plot the observed t-statistic
plt.axvline(
    x=t_statistic_student,
    color='limegreen', linestyle='-.',
    label=f't ({t_statistic_student:.3f})')

# Shade the P-value areas (two-tailed)
plt.fill_between(
    x[x <= -abs(t_statistic_student)],
    hx[x <= -abs(t_statistic_student)],
    color='greenyellow',
    label=f'P ({p_value_student:.3f})',)
plt.fill_between(
    x[x >= abs(t_statistic_student)],
    hx[x >= abs(t_statistic_student)],
    color='greenyellow')

# Add labels and title
plt.xlabel(r'$t$')
plt.ylabel('Density')
plt.margins(x=0, y=0)
plt.yticks([])
plt.title(f"t-distribution (DF={df_student})")
plt.legend();

#### Student's confidence interval


In [ ]:
# Calculate the confidence interval (e.g., 95% confidence)
margin_of_error_student = abs(
    stats.t.ppf((1 + (1 - α)) / 2, df_student)) * se_pooled

ci_student = np.array([
    mean_diff - margin_of_error_student,
    mean_diff + margin_of_error_student])

# Print the results
print(
    "95% confidence interval for the mean difference:",
    ci_student.round(3))

#### Performing the Student's t-test in Python


In [ ]:
# Student's t-test using SciPy (assuming equal variances)
t_statistic_scipy_student, p_value_scipy_student = stats.ttest_ind(
    yng, old,
    equal_var=True)

# Student's t-test using statsmodels (assuming equal variances)
t_statistic_sm_student, p_value_sm_student, df_sm_student = \
sm.stats.ttest_ind(yng, old, usevar='pooled')

# Student's t-test using Pingouin (assuming equal variances)
ttest_results_pingouin_student = pg.ttest(
    yng, old,
    correction=False)

# Print the results
print("Student's t-test results (equal variances):")
print(f"t-statistic (SciPy) = {t_statistic_scipy_student:.3f}, \
P value = {p_value_scipy_student:.4f}")
print(f"t-statistic (statsmodels) = {t_statistic_sm_student:.3f}, \
P value = {p_value_sm_student:.4f}, DF = {df_sm_student:n}")
print()
print("Student's t-test summary table (Pingouin):")
print(ttest_results_pingouin_student.round(3))

### Effect size


In [ ]:
# Calculate Cohen's d manually
cohen_denominator = np.sqrt(
    ((n_yng - 1) * var_yng + (n_old - 1) * var_old) / (n_yng+n_old-2)
)
cohens_d_manual = mean_diff / cohen_denominator

# Calculate unbiased Cohen's d using pingouin
effect_size_pingouin = pg.compute_effsize(yng, old, eftype='cohen')

# Print the results
print(f"Cohen's d (manual): {cohens_d_manual:.3f}")
print(f"Unbiased Cohen's d (pingouin): {effect_size_pingouin:.3f}")

### Working with real-world data
#### Loading the data


In [ ]:
import pandas as pd

# Load the 'penguins' dataset
penguins = pg.read_dataset('penguins')

# Display the first 5 rows of the dataset
print(penguins.head())

#### Understanding the data


In [ ]:
from tabulate import tabulate

# Display descriptive statistics
print(
    tabulate(
        penguins.describe(include='all'),
        tablefmt='simple',
        headers='keys',  # Use column indices as headers
        showindex=True,
        floatfmt='.1f'))

In [ ]:

sns.pairplot(penguins, hue="species", diag_kind="hist", height=3)
plt.legend(loc='lower center');

#### Checking assumptions


In [ ]:
# Statistics for groups of species
print(
    penguins
    .groupby(by='species')[['flipper_length_mm', 'bill_depth_mm']]
    .describe()
    .round(2))

In [ ]:
#| fig-subcap: 
#|   - "Distribution of flipper length (mm)"
#|   - "Distribution of bill depth (mm)"
#| layout-ncol: 2

# Create a copy of the original DataFrame while removing Chinstrap species
penguins_filtered = penguins[penguins['species'] != 'Chinstrap'].copy()
# penguins_filtered = penguins[
#     penguins['species'].isin(['Adelie', 'Gentoo'])].copy()

# # Box and swarm plots for flipper length
plt.figure(figsize=(3.5, 3))
sns.boxplot(
    data=penguins_filtered,
    x='species', y='flipper_length_mm',
    hue='species')
sns.swarmplot(
    data=penguins_filtered,
    x='species', y='flipper_length_mm',
    color='black')

plt.xlabel('Species')
plt.ylabel('Flipper length (mm)')
plt.title("Flipper length")
plt.show();

# Box and swarm plots for bill depth
plt.figure(figsize=(3.5, 3))
sns.boxplot(
    data=penguins_filtered,
    x='species', y='bill_depth_mm',
    hue='species')
sns.swarmplot(
    data=penguins_filtered,
    x='species', y='bill_depth_mm',
    color='black')

# Highlight overlapping
plt.axhspan(15.25, 18.25, facecolor='red', alpha=0.25, edgecolor=None)

plt.xlabel('Species')
plt.ylabel('Bill depth (mm)')
plt.title("Bill depth")
plt.show();

In [ ]:

# Plotting all Q-Q plots in 2x2
_, axes = plt.subplots(nrows=2, ncols=2, figsize=(6, 6))
for row, variable in enumerate(('flipper_length_mm', 'bill_depth_mm')):
    for col, species in enumerate(('Adelie', 'Gentoo')):
        pg.qqplot(
            penguins_filtered.loc[
                penguins_filtered['species'] == species, variable].values,
            ax=axes[row, col],
        )
        axes[row, col].set_title(f"{variable}, {species}")

plt.tight_layout();

In [ ]:
# Perform tests using ouor custom function

for variable in ('flipper_length_mm', 'bill_depth_mm'):
    for species in ('Adelie', 'Gentoo'):
        normality_tests(
            penguins_filtered.loc[
                penguins_filtered['species'] == species, variable],
            f"{variable} in {species}")
        print()  # Print a blank line space

In [ ]:
# Levene's test for equal variances

for variable in ('flipper_length_mm', 'bill_depth_mm'):
    print(f"Homoscedasticity test for {variable}:")
    print(
        pg.homoscedasticity(
            data=penguins_filtered,
            dv=variable,
            group='species',
            method='levene'))
    print()

#### The problem of missing values


In [ ]:
penguins_filtered.info()

In [ ]:
# Create a clean copy of the dataset with NaN values filtered out
penguins_filtered_clean = penguins_filtered.dropna(
    subset=['flipper_length_mm', 'bill_depth_mm'])

# Levene test with the clean dataset
for variable in ('flipper_length_mm', 'bill_depth_mm'):
    print(f"Homoscedasticity test for {variable}:")
    print(
        pg.homoscedasticity(
            data=penguins_filtered_clean,
            dv=variable,
            group='species',
            method='levene').round(3))
    print()

#### Performing t-tests


In [ ]:
# Perform Student t-test for flipper length
variable = 'flipper_length_mm'
print(f"Student's t-test for {variable}:")
print(
    flipper_ttest_pg := pg.ttest(
        x=penguins.loc[penguins["species"] == 'Adelie', variable],
        # pg.ttest removes missing values by default
        y=penguins.loc[penguins["species"] == 'Gentoo', variable],
        correction=False))

#### Overlapping distributions


In [ ]:
# Perform Student t-test for bill depth
variable = 'bill_depth_mm'
print(f"Student's t-test for {variable}:")
print(
    pg.ttest(
        x=penguins.loc[penguins["species"] == 'Adelie', variable],
        y=penguins.loc[penguins["species"] == 'Gentoo', variable],
        correction=False))

#### One-sided t-test


In [ ]:
# Perform Student's t-test using statsmodels
variable = 'bill_depth_mm'
ttest_results_sm_penguins = sm.stats.ttest_ind(
    x1=penguins_filtered_clean.loc[
        penguins_filtered_clean["species"] == 'Adelie', variable],
    # !statsmodels doesn't support policy for missing values
    x2=penguins_filtered_clean.loc[
        penguins_filtered_clean["species"] == 'Gentoo', variable],
    usevar='pooled',
    alternative='larger',  # is Adelie bill depth mean LARGER than Gentoo
    value=3                # Difference between the means under H0
)

# Unpack and print the relevant statistics
t_statistic_sm_penguins, p_value_sm_penguins, df_sm_penguins = \
ttest_results_sm_penguins

print(f"One-sided Student's t-test on {variable} (statsmodels):")
print(f" t-statistic = {t_statistic_sm_penguins:.3f}\n\
 P value = {p_value_sm_penguins:.4f}\n DF= {df_sm_penguins:n}")

In [ ]:
# One-sided Student t-test using Pingouin
print(f"One-sided Student's t-test on {variable} (Pingouin):")
print(
    pg.ttest(
        x=penguins.loc[penguins["species"] == 'Adelie', variable] - 3,
        y=penguins.loc[penguins["species"] == 'Gentoo', variable],
        alternative='greater',  # One-sided t-test for x > y
        correction=False).round(4))

## Non-parametric methods


### Mann-Whitney U test
#### Exact method


In [ ]:
from scipy.stats import rankdata

# Combine the data for ranking
combined_data = np.concatenate((old, yng))

# Rank the combined data using the average method (default) for ties
ranks = rankdata(combined_data, method='average')

# Separate the ranks back into the original groups
old_ranks = ranks[:len(old)]
yng_ranks = ranks[len(old):]

# Calculate the sum of ranks for each group
R_old = np.sum(old_ranks)
R_yng= np.sum(yng_ranks)

# Number of data points in each group
n_old = len(old)
n_yng = len(yng)

# Calculate the U-statistic for each group
U_old = n_old * n_yng + (n_old * (n_old + 1)) / 2 - R_old
U_yng = n_old * n_yng + (n_yng * (n_yng + 1)) / 2 - R_yng

# The Mann-Whitney U-statistic is the minimum of U_old and U_yng
U_statistic = min(U_old, U_yng)

# Critical U value (two-tailed) from table, for n1=9, n2=8, alpha=0.05
# https://real-statistics.com/statistics-tables/mann-whitney-table/
U_critical = 15

# Print the results
print("Mann-Whitney U test - Manual calculation\n")

print("1. Combined and ranked data:")
print("  Combined data:", combined_data)
print("  Corresponding ranks:", ranks)
print()
print("2. Rank sums:")
print("  Sum of ranks for 'old' group =", R_old)
print("  Sum of ranks for 'yng' group =", R_yng)
print()
print("3. U-statistics calculation:")
print("  Number of observations in 'old' group =", n_old)
print("  Number of observations in 'yng' group =", n_yng)
print()
print("  U-statistic for 'old' group (U_old) =", U_old)
print("  U-statistic for 'yng' group (U_yng) = ", U_yng)
print(f"  Mann-Whitney U-statistic (min(U_old, U_yng)) = {U_statistic}")
print()
print("4. Comparison with critical value:")
print(f"  Critical U value (n={n_yng}, m={n_old}, \
alpha=0.05, two-tailed): {U_critical}\n")

# Compare U to the critical value and draw a conclusion
if U_statistic <= U_critical:
    print("Conclusion: reject the null hypothesis.")
    print("  There is a statistically significant difference \
between the 'old' and 'yng' groups.")
else:
    print("Conclusion: fail to reject the null hypothesis.")
    print("  There is no statistically significant difference \
between the 'old' and 'yng' groups.")

In [ ]:
def mann_whitney_u_prob(U, n, m):
    """
    Calculate the probability mass function of a specific U
    value in the Mann-Whitney U distribution.

    Args:
        U: the U-statistic value
        n: the number of observations in the first group
        m: the number of observations in the second group

    Returns:
        The probability of the U value.
    """

    if n == 0 or m == 0:
        return 1 if U == 0 else 0
    elif n > 0 and m > 0:
        return (
            n * mann_whitney_u_prob(U - m, n - 1, m)  # Recursive call
            + m * mann_whitney_u_prob(U, n, m - 1)) / (n + m)
    else:
        return 0  # Handle invalid input (negative n or m)

In [ ]:
# Generate the table for n = 8
n = n_yng
U_values = range(10)     # U from 0 to 9
m_values = range(1, 10)  # m from 1 to 9

probability_distribution_U = {}
for m in m_values:
    row_data = {}
    for U in U_values:
        row_data[U] = mann_whitney_u_prob(U, n, m)
    probability_distribution_U[m] = row_data

probability_distribution_U = pd.DataFrame(probability_distribution_U)
probability_distribution_U.columns.name="m"
probability_distribution_U.index.name="U"

# Display the table
print(f"Probability distribution (PMF) of the U-statistic for n={n}:\n")
print(probability_distribution_U.round(4))

In [ ]:
# Calculate the cumulative sum of probabilities along the rows (axis=0)
print(f"Cumulative sum of probabilities (CDF) of the U-statistic \
for n={n}:\n")
print(probability_distribution_U.cumsum(axis=0).round(4))

In [ ]:
# Display the cumulative probability for U=7 and m=9 (n=8)
cumulative_prob_U = \
probability_distribution_U.cumsum(axis=0).loc[U_statistic, m].item()
# Multiply by 2 for two-tailed test
p_value_U_manual = 2 * cumulative_prob_U

print(f"Cumulative probability for U={U_statistic}, n={n} and m={m}: \
{cumulative_prob_U:.5f}")
print(f"Associated P value (two-tailed): {p_value_U_manual:.5f}")

# Compare the P value to 0.05
if p_value_U_manual < 0.05:
    print("Reject the null hypothesis.")
    print(" There is a significant difference between the groups.")
else:
    print("Fail to reject the null hypothesis.")
    print(" There is no significant difference between the groups.")

#### Approximating the U-statistic distribution


In [ ]:

from scipy.stats import norm

# Determine the mean and SD to approximate the distribution
μ = n_old * n_yng / 2
σ = np.sqrt(n_old * n_yng * (n_old + n_yng + 1) / 12)

# Generate x values for plotting the normal distribution
x = np.linspace(μ - 4 * σ, μ + 4 * σ, 1000)
normal_pdf = norm.pdf(x, loc=μ, scale=σ)

# Create the plot
plt.figure(figsize=(3.5, 3))
plt.plot(x, normal_pdf, lw=2, color="black")

# Plot the critical U-values (two-tailed test)
U_crit_lower = μ - norm.ppf(1 - α/2) * σ
U_crit_upper = μ + norm.ppf(1 - α/2) * σ

# Calculate the z-score
z_score = (U_statistic - μ) / σ

# Calculate the p-value (two-tailed)
p_value_U = 2 * (1 - norm.cdf(abs(z_score)))

plt.axvline(
    x=U_crit_lower,
    color='orangered', linestyle='--')
plt.axvline(
    x=U_crit_upper,
    color='orangered', linestyle='--',
    label=f"U* ({U_crit_lower:.3f})")

# Shade the rejection regions (alpha)
plt.fill_between(
    x, normal_pdf,
    where=(x <= U_crit_lower) | (x >= U_crit_upper),
    linestyle="-", linewidth=2, color='tomato',
    label=f'α ({α})')

# Plot the observed U-statistic
plt.axvline(
    x=float(U_statistic),
    color='limegreen', linestyle='-.',
    label=f'U ({U_statistic:n})')

# Shade the P-value areas (two-tailed)
plt.fill_between(
    x, normal_pdf,
    where=(x <= U_statistic) | (x >= μ + (μ - U_statistic)),
    color='greenyellow',
    label=f"P ({p_value_U:.4f})")

# Add labels and title
plt.xlabel('U')
plt.ylabel('Probability density')
plt.margins(x=0, y=0)
plt.yticks([])
plt.title("Normal approximation to U")
plt.legend();

### Python tools for non-parametric unpaired t-test


In [ ]:
from scipy.stats import mannwhitneyu

# Mann-Whitney U test with exact P value calculation
mwu_statistic_exact_scipy, mwu_p_value_exact_scipy = mannwhitneyu(
    old, yng, method='exact')

# Mann-Whitney U test with asymptotic (normal approximation) 
# P value calculation
mwu_statistic_asymptotic_scipy, mwu_p_value_asymptotic_scipy = mannwhitneyu(
    old, yng, method='asymptotic', use_continuity=False)

# Print the results
print("Mann-Whitney U test results (SciPy):")
print(f" P value (exact method) = {mwu_p_value_exact_scipy:.4f}")
print(f" P value (asymptotic) = {mwu_p_value_asymptotic_scipy:.4f}")

In [ ]:
print("Mann-Whitney U test results (Pingouin):")
print(
    pg.mwu(
        x=old,
        y=yng,
        method='asymptotic',
        use_continuity=False,
        alternative='two-sided').round(4))

#### Continuity correction


In [ ]:
print("Mann-Whitney U test results (with continuity correction):")
print(
    pg.mwu(
        old, yng,
        method='asymptotic',
        use_continuity=True,))

## Bootstrapping and permutation tests
### The essence of bootstrapping


In [ ]:
# Set random seed for reproducibility
np.random.seed(111)

# Generate 10,000 bootstrap replicates of the mean for each group
# !Make sure to choose exactly N elements randomly **with replacement**
n_replicates = 10000
bs_old_means = np.array([
    np.mean(
        np.random.choice(
            old,
            size=len(old),
            replace=True
        )) for _ in range(n_replicates)])

bs_yng_means = np.array([
    np.mean(
        np.random.choice(
            yng,
            size=len(yng),
            replace=True)) for _ in range(n_replicates)])

# Calculate the difference between the bootstrap means for each 
# replicate using NumPy's broadcast operation
bs_mean_diff = bs_yng_means - bs_old_means

print("First ten calculated bootstrap differences between group means:")
print(bs_mean_diff[:10].round(3))

### Estimate and percentile interval


In [ ]:
# Calculate the 95% percentile interval using np.percentile
bs_m = np.mean(bs_mean_diff)
bs_ci = np.percentile(bs_mean_diff, [2.5, 97.5])

# Print the results
print(f"Bootstrap estimate of the mean difference = {bs_m:5.2f}")
print(r"95% bootstrap percentile interval: ", bs_ci.round(3))

In [ ]:

plt.figure(figsize=(3.5, 3))

# Plot the histogram of the bootstrap distribution
plt.hist(
    bs_mean_diff,
    density=False,
    bins='auto',  # or use bins=int(n_replicates**.5)
    color='gold',
    label='Bootstrap replicates')

# Annotate the observed mean difference
plt.axvline(
    x=mean_diff.item(),
    color='cornflowerblue', linestyle='-', lw=2,
    label=f'Observed estimate ({mean_diff:.3f})')

# Add lines for the percentile interval
plt.axvline(
    x=bs_ci[0],
    color='red', linestyle='--',
    label=f'2.5th ({bs_ci[0]:.2f})')
plt.axvline(
    x=bs_ci[1],
    color='red', linestyle='-.',
    label=f'97.5th ({bs_ci[1]:.2f})')

# Add labels and title
plt.xlabel('Mean difference')
plt.ylabel('Frequency')
plt.title(f"Bootstrap mean differences")
plt.legend(fontsize=9);

In [ ]:
# Calculate the 95% normal margin of error
bs_s = np.std(bs_mean_diff, ddof=1)
z_crit = norm.ppf(1 - α/2)
bs_w =  z_crit * bs_s

# Calculate confidence interval
bs_ci_normal = np.array([bs_m - bs_w, bs_m + bs_w])

# Print the results
print(f"Bootstrap standard error = {bs_s:.4f}")
print("95% asymptotic CI of bootstrap distribution: ",bs_ci_normal.round(3))

### Bootstrapping with pingouin


In [ ]:
# Calculate 95% percentile bootstrap confidence interval using Pingouin
bs_ci_pg, bt_pg = pg.compute_bootci(
    x=yng,
    y=old,
    func=lambda x,y : np.mean(x) - np.mean(y),
    confidence=0.95,
    method='per',
    seed=111,
    decimals=3,
    n_boot=10000,
    return_dist=True)

print(
    "First eight calculated bootstrap differences group means (Pingouin):")
print(bt_pg.round(2)[:8])
print()
print("95% bootstrap percentile interval (pingouin): ", bs_ci_pg.round(3))

### Bootstrap P value
#### Shifted t-statistics
#### The 99 rule


In [ ]:
# Set the random seed for reproducibility
np.random.seed(111)

n_replicates = 10**4 - 1  # 99 rule

# Calculate the combined mean of both groups
combined_mean = np.mean(np.concatenate([old, yng]))
print("Combined mean of both groups = ", combined_mean.round(3))

# Shift the data in each group to have the same mean (combined_mean)
old_shifted = old - np.mean(old) + combined_mean
yng_shifted = yng - np.mean(yng) + combined_mean

# Generate B bootstrap replicates for each shifted group
bs_shifted_old = np.array([
    np.random.choice(
        old_shifted,
        size=len(old),
        replace=True) for _ in range(n_replicates)])
bs_shifted_yng = np.array([
    np.random.choice(
        yng_shifted,
        size=len(yng),
        replace=True) for _ in range(n_replicates)])

# Calculate sample means and variances for each shifted replicate
# !np.mean and np.var calculate mean and variance of the entire flattened 
# array by default if no 'axis' is provided, so we need to calculate mean 
# and var along each row/replicate
bs_shifted_old_means = np.mean(bs_shifted_old, axis=1)
bs_shifted_yng_means = np.mean(bs_shifted_yng, axis=1)

bs_shifted_old_vars = np.var(bs_shifted_old, ddof=1, axis=1)
bs_shifted_yng_vars = np.var(bs_shifted_yng, ddof=1, axis=1)

# Calculate SE for the difference means for each replicate using 
# Welch's formula
bs_se_unequal = np.sqrt(
    bs_shifted_old_vars / len(old)
    + bs_shifted_yng_vars / len(yng))

# Calculate Welch's t-statistic for each shifted replicate
bs_t_statistic_welch = (
    bs_shifted_yng_means - bs_shifted_old_means
) / bs_se_unequal

# Or simply perform Welch's t-test on the shifted replicates using list comprehension
# bs_t_statistic_welch = np.array([
#     stats.ttest_ind(
#         x, y, equal_var=False)[0]
#     for x, y in zip(bs_shifted_yng, bs_shifted_old)])

print()
print("First ten calculated shifted t-statistics:")
print(bs_t_statistic_welch.round(2)[:10])

#### One-tailed shifted P value


In [ ]:
# Calculate one-sided P value using shifting, considering the 
# direction of the observed statistic.
# 1. Calculate lower and upper tail
if t_statistic_welch >= 0:
    # p_value_bs_shifted_low=np.sum(bs_t_statistic_welch>=t_statistic_welch)
    # / len(bs_t_statistic_welch)
    p_value_bs_shifted_up = np.mean(
        bs_t_statistic_welch >= t_statistic_welch)
    p_value_bs_shifted_low = np.mean(
        bs_t_statistic_welch <= -t_statistic_welch)
else:
    p_value_bs_shifted_low = np.mean(
        bs_t_statistic_welch <= t_statistic_welch)
    p_value_bs_shifted_up = np.mean(
        bs_t_statistic_welch >= -t_statistic_welch)

# 2. Attribute the one-tailed P value
p_value_bs_shifted_1t = p_value_bs_shifted_up if t_statistic_welch >= 0 \
else p_value_bs_shifted_low

# Print the P value
print(f"One-sided P value using bootstrap shifted t-statistics = \
{p_value_bs_shifted_1t:.4f}")

In [ ]:

import matplotlib.patches as mpatches

plt.figure(figsize=(3.5, 3))

# Plot the histogram of the shifted t-statistics
hist, bins, patches = plt.hist(
    bs_t_statistic_welch,
    density=False,
    bins=int(n_replicates**.5),
    color='gold')

# Annotate the observed t-statistic
plt.axvline(
    x=t_statistic_welch,
    color='limegreen', linestyle='-.', lw=2,
    label=f'Observed t ({t_statistic_welch:.3f})')

# Determine the direction of the observed statistic and plot accordingly
if t_statistic_welch >= 0:
    # Plot the histogram of the shifted t-statistics 
    # >= observed t-statistic
    extreme_t_stats = bs_t_statistic_welch[
        bs_t_statistic_welch >= t_statistic_welch]
    # Change the color of the bars based on the direction parameter
    for i, bin_edge in enumerate(bins[:-1]):
        if np.any(bin_edge >= extreme_t_stats):
            patches[i].set_facecolor('lime')

else:
    # Plot the histogram of the shifted t-statistics <= observed t-statistic
    extreme_t_stats = bs_t_statistic_welch[
        bs_t_statistic_welch <= t_statistic_welch]
    for i, bin_edge in enumerate(bins[:-1]):
        if np.any(bin_edge <= extreme_t_stats):
            patches[i].set_facecolor('lime')

# Add labels and title
plt.xlabel('t')
plt.ylabel('Frequency')
plt.title(f"Shifted Welch t under H0")

# Create a copy of the original patch for the legend
original_patch = mpatches.Patch(color='gold', label='Bootstrap t')
# Create a patch for the legend
p_value_patch = mpatches.Patch(
    color='lime', label=f'One-tailed P ({p_value_bs_shifted_1t:.4f})')

# Add the patches to the legend
plt.legend(handles=[original_patch, plt.gca().lines[0], p_value_patch]);

#### Estimating two-tailed shifted P value


In [ ]:
# Maximum one-tail
p_value_bs_shifted_2t_max = 2 * max(
    p_value_bs_shifted_low, p_value_bs_shifted_up)

# Sum of the tails
p_value_bs_shifted_2t_sum = p_value_bs_shifted_low + p_value_bs_shifted_up

# Print the results
print(f"Two-tailed t-test bootstrap (shifted) P value (conservative) = \
{p_value_bs_shifted_2t_max:.4f}")
print(f"Two-tailed t-test bootstrap (shifted) P value (both tails) = \
{p_value_bs_shifted_2t_sum:.4f}")

#### Permutation


In [ ]:
def permutation_sample(data1, data2):
    """Generate a permutation sample by combining and shuffling two datasets

    This function concatenates `data1` and `data2`, randomly shuffles the
    combined array, and then splits it back into two arrays with the 
    original sizes of `data1` and `data2`, respectively.

    Args:
        data1: a one-dimensional array representing the first dataset
        data2: a one-dimensional array representing the second dataset

    Returns:
        A tuple containing two NumPy arrays:
            - permuted_data1: a shuffled version of `data1`
            - permuted_data2: a shuffled version of `data2`
    """

    # Combine the two datasets into a single array
    combined_data = np.concatenate([data1, data2])

    # Shuffle the combined data in-place
    np.random.shuffle(combined_data)

    # Split the shuffled data back into two arrays with original sizes
    permuted_data1 = combined_data[:len(data1)]
    permuted_data2 = combined_data[len(data1):]

    return permuted_data1, permuted_data2

In [ ]:
# Set random seed for reproducibility
np.random.seed(111)

# Generate 9999 bootstrap replicates
n_replicates = 10**4 - 1  # 99 rule

# Initialize an empty array to store the bootstrap mean differences
permuted_mean_diffs = np.zeros(n_replicates)

# Bootstrap replicates of the mean difference for each permutation sample
for i in range(n_replicates):
    # Generate a new permuted sample
    x, y = permutation_sample(old, yng)

    # Calculate the mean difference for this permuted sample
    permuted_mean_diffs[i] = np.mean(x) - np.mean(y)

# We could also use a list comprehension instead of the loop
# permuted_mean_diffs = np.array([
#     np.mean(x) - np.mean(y) for _ in range(n_replicates) for x, y in [
#         permutation_sample(old, yng)]])

print("Firt ten mean differences of the permutation replicates:")
print(permuted_mean_diffs.round(2)[:10])

In [ ]:
# Calculate one-sided P value using the permutation distribution 
# of mean differences
if mean_diff >= 0:
    p_value_permutation_up = np.mean(permuted_mean_diffs >= mean_diff)
    p_value_permutation_low = np.mean(permuted_mean_diffs <= -mean_diff)
else:
    p_value_permutation_low = np.mean(permuted_mean_diffs <= mean_diff)
    p_value_permutation_up = np.mean(permuted_mean_diffs >= -mean_diff)

p_value_permutation_1t = p_value_permutation_up if mean_diff >= 0 \
else p_value_permutation_low

# Print the P value
print(f"One-sided P value using permutation test = \
{p_value_permutation_1t:.4f}")

In [ ]:

plt.figure(figsize=(3.5, 3))

# Plot the histogram of the shuffled mean differences
hist, bins, patches = plt.hist(
    permuted_mean_diffs,
    density=False,
    bins=int(n_replicates**.5),
    color='gold')

# Annotate the observed mean difference
plt.axvline(
    x=mean_diff,
    color='limegreen', linestyle='-.', lw=2,
    label=f'Observed estimate ({mean_diff:.3f})')

# Determine the direction of the observed difference and plot accordingly
if mean_diff >= 0:
    # histogram of the mean differences >= observed mean difference 
    extreme_diffs = permuted_mean_diffs[permuted_mean_diffs >= mean_diff]
    # Change the color of the bars based on the direction parameter
    for i, bin_edge in enumerate(bins[:-1]):
        if np.any(bin_edge >= extreme_diffs):
            patches[i].set_facecolor('lime')
else:
    # histogram of the mean differences <= observed mean difference
    extreme_diffs = permuted_mean_diffs[permuted_mean_diffs <= mean_diff]
    for i, bin_edge in enumerate(bins[:-1]):
        if np.any(bin_edge <= extreme_diffs):
            patches[i].set_facecolor('lime')

# Add labels and title
plt.xlabel('Mean difference')
plt.ylabel('Frequency')
plt.title(f"Permutation distribution under H0")

# Create a copy of the original patch for the legend
original_patch = mpatches.Patch(color='gold', label='Permuted replicates')
# Create a patch for the legend
p_value_patch = mpatches.Patch(
    color='lime', label=f'One-tailed P ({p_value_permutation_1t:.4f})')

# Add the patches to the legend
plt.legend(handles=[original_patch, plt.gca().lines[0], p_value_patch]);

In [ ]:
# Maximum one-tail
p_value_permut_2t_max = 2 * max(
    p_value_permutation_low, p_value_permutation_up)

# Sum of the tails
p_value_permut_2t_sum = p_value_permutation_low + p_value_permutation_up

# Print the results
print(f"Two-tailed permutation P value (conservative) = \
{p_value_permut_2t_max:.4f}")
print(f"Two-tailed permutation P value (both tails) = \
{p_value_permut_2t_sum:.4f}")

## Conclusion
